# 🔬 v6 Model Forensic & Safety Analysis

**Goal:** Verify clinical safety of `xgboost_v6_pure_final.json`

**Analysis:**
1. Global Performance Audit
2. Clinical Safety Stress Test
3. Logic Visualization
4. Deep Dive on Errors

## 1. Setup

In [ ]:
!pip install polars xgboost seaborn -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
print(f"Polars: {pl.__version__}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/processed"

FILES = {
    "model": f"{BASE_PATH}/xgboost_v6_pure_final.json",
    "test": f"{BASE_PATH}/master_test_v6_engineered.parquet"
}

for name, path in FILES.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"{exists} {name}: {path}")

## 2. Load Model & Data

In [ ]:
# Load model
print("📂 Loading model...")
model = xgb.Booster()
model.load_model(FILES["model"])
print("✅ Model loaded")

In [ ]:
# Features
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]
TARGET = 'bolus'

# Load test data
print("\n📂 Loading test data...")
df = pl.read_parquet(FILES["test"])
print(f"   Rows: {len(df):,}")
print(f"   Columns: {df.columns}")

In [ ]:
# Prepare features
available = [f for f in FEATURES if f in df.columns]
print(f"\n📋 Available features: {len(available)}")

# Fill nulls
for col in available:
    df = df.with_columns(pl.col(col).fill_null(0))

# Create DMatrix
X = df.select(available).to_numpy().astype(np.float32)
y = df[TARGET].to_numpy().astype(np.float32)
dtest = xgb.DMatrix(X, feature_names=available)

print(f"   X shape: {X.shape}")

In [ ]:
# Generate predictions
print("🔮 Generating predictions...")
preds = model.predict(dtest)

# Add to dataframe
df = df.with_columns([
    pl.Series("pred_bolus", preds),
    pl.Series("error", preds - y)
])

print(f"✅ Predictions added")

## 3. Global Performance Audit

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y, preds)
rmse = np.sqrt(mean_squared_error(y, preds))

# Positive-only
pos_mask = y > 0
pos_mae = mean_absolute_error(y[pos_mask], preds[pos_mask]) if pos_mask.sum() > 0 else 0

print("📊 GLOBAL PERFORMANCE")
print("=" * 50)
print(f"   Total Rows: {len(df):,}")
print(f"   Global MAE: {mae:.4f}")
print(f"   Global RMSE: {rmse:.4f}")
print(f"   Positive-Only MAE: {pos_mae:.4f}")

In [ ]:
# Residual histogram
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
errors = df['error'].to_numpy()
plt.hist(errors, bins=100, color='steelblue', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', label='Zero Error')
plt.axvline(errors.mean(), color='orange', linestyle='--', label=f'Mean: {errors.mean():.3f}')
plt.xlabel('Error (Pred - True)')
plt.ylabel('Count')
plt.title('Error Distribution')
plt.legend()
plt.xlim(-5, 5)

plt.subplot(1, 2, 2)
plt.hist(preds, bins=100, color='coral', edgecolor='black', alpha=0.7)
plt.xlabel('Predicted Bolus')
plt.ylabel('Count')
plt.title('Prediction Distribution')
plt.xlim(0, 10)

plt.tight_layout()
plt.show()

## 4. Clinical Safety Stress Test

In [ ]:
print("🏥 CLINICAL SAFETY STRESS TEST")
print("=" * 60)

# Filter no-carbs scenarios
df_no_carbs = df.filter(pl.col('carbs') == 0)

# 1. Hypo Zone (<70)
hypo = df_no_carbs.filter(pl.col('CGM') < 70)
hypo_mean = hypo['pred_bolus'].mean() if len(hypo) > 0 else 0
hypo_max = hypo['pred_bolus'].max() if len(hypo) > 0 else 0
hypo_safe_pct = len(hypo.filter(pl.col('pred_bolus') < 0.1)) / len(hypo) * 100 if len(hypo) > 0 else 0

print(f"\n🔴 HYPO ZONE (<70 mg/dL, No Carbs): {len(hypo):,} rows")
print(f"   Mean Prediction: {hypo_mean:.4f} U")
print(f"   Max Prediction: {hypo_max:.4f} U")
print(f"   Safe (<0.1 U): {hypo_safe_pct:.1f}%")
status = "✅ SAFE" if hypo_mean < 0.2 else "⚠️ NEEDS HARD RULE"
print(f"   Status: {status}")

In [ ]:
# 2. Safety Zone (70-120)
safety = df_no_carbs.filter((pl.col('CGM') >= 70) & (pl.col('CGM') < 120))
safety_mean = safety['pred_bolus'].mean() if len(safety) > 0 else 0
safety_safe_pct = len(safety.filter(pl.col('pred_bolus') < 0.2)) / len(safety) * 100 if len(safety) > 0 else 0

print(f"\n🟡 SAFETY ZONE (70-120 mg/dL, No Carbs): {len(safety):,} rows")
print(f"   Mean Prediction: {safety_mean:.4f} U")
print(f"   Safe (<0.2 U): {safety_safe_pct:.1f}%")
status = "✅ SAFE" if safety_safe_pct > 90 else "⚠️ NEEDS REVIEW"
print(f"   Status: {status}")

In [ ]:
# 3. High Zone (>250)
high = df_no_carbs.filter(pl.col('CGM') > 250)
high_mean = high['pred_bolus'].mean() if len(high) > 0 else 0

if len(high) > 0:
    high_corr = np.corrcoef(high['CGM'].to_numpy(), high['pred_bolus'].to_numpy())[0, 1]
else:
    high_corr = 0

print(f"\n🟢 HIGH ZONE (>250 mg/dL, No Carbs): {len(high):,} rows")
print(f"   Mean Prediction: {high_mean:.4f} U")
print(f"   CGM-Bolus Correlation: {high_corr:.3f}")
status = "✅ GOOD" if high_corr > 0.3 and high_mean > 0.5 else "⚠️ WEAK RESPONSE"
print(f"   Status: {status}")

In [ ]:
# Summary table
print("\n" + "=" * 60)
print("📋 CLINICAL SAFETY SUMMARY")
print("=" * 60)
print(f"{'Zone':20} | {'Rows':>10} | {'Mean Pred':>10} | {'Status':>15}")
print("-" * 60)
print(f"{'Hypo (<70)':20} | {len(hypo):>10,} | {hypo_mean:>10.4f} | {'✅ SAFE' if hypo_mean < 0.2 else '⚠️ DANGER'}")
print(f"{'Safety (70-120)':20} | {len(safety):>10,} | {safety_mean:>10.4f} | {'✅ SAFE' if safety_safe_pct > 90 else '⚠️ REVIEW'}")
print(f"{'High (>250)':20} | {len(high):>10,} | {high_mean:>10.4f} | {'✅ GOOD' if high_mean > 0.5 else '⚠️ WEAK'}")

## 5. Visualizing the Logic

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Zero-Line Scatter: CGM vs pred_bolus
ax1 = axes[0, 0]
sample_size = min(50000, len(df))
sample = df.sample(sample_size, seed=42)
ax1.scatter(sample['CGM'].to_numpy(), sample['pred_bolus'].to_numpy(), 
            alpha=0.1, s=1, c='steelblue')
ax1.axhline(0, color='red', linestyle='--', alpha=0.7)
ax1.axvline(70, color='orange', linestyle='--', alpha=0.7, label='Hypo Threshold')
ax1.axvline(120, color='green', linestyle='--', alpha=0.5)
ax1.set_xlabel('CGM (mg/dL)')
ax1.set_ylabel('Predicted Bolus (U)')
ax1.set_title('CGM vs Prediction (50K samples)')
ax1.set_xlim(40, 400)
ax1.set_ylim(-0.5, 10)

# 2. Feature Importance
ax2 = axes[0, 1]
importance = model.get_score(importance_type='gain')
sorted_imp = sorted(importance.items(), key=lambda x: x[1], reverse=True)
features_sorted = [x[0] for x in sorted_imp[:12]]
values_sorted = [x[1] for x in sorted_imp[:12]]
ax2.barh(features_sorted[::-1], values_sorted[::-1], color='teal')
ax2.set_xlabel('Gain')
ax2.set_title('Feature Importance (Top 12)')

# 3. Trend Sensitivity: cgm_delta vs error
ax3 = axes[1, 0]
if 'cgm_delta_15min' in df.columns:
    delta_sample = sample.filter(pl.col('cgm_delta_15min').abs() < 50)
    ax3.scatter(delta_sample['cgm_delta_15min'].to_numpy(), 
                delta_sample['error'].to_numpy(),
                alpha=0.1, s=1, c='coral')
    ax3.axhline(0, color='black', linestyle='--')
    ax3.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_xlabel('CGM Delta 15min')
    ax3.set_ylabel('Error (Pred - True)')
    ax3.set_title('Trend Sensitivity: Does model over-predict on falling BG?')
else:
    ax3.text(0.5, 0.5, 'cgm_delta_15min not available', ha='center')

# 4. Error by CGM bin
ax4 = axes[1, 1]
cgm_bins = [(0, 70), (70, 100), (100, 140), (140, 180), (180, 250), (250, 500)]
bin_labels = ['<70', '70-100', '100-140', '140-180', '180-250', '>250']
bin_errors = []
for low, high in cgm_bins:
    subset = df.filter((pl.col('CGM') >= low) & (pl.col('CGM') < high))
    if len(subset) > 0:
        bin_errors.append(subset['error'].abs().mean())
    else:
        bin_errors.append(0)
ax4.bar(bin_labels, bin_errors, color='purple', alpha=0.7)
ax4.set_xlabel('CGM Range (mg/dL)')
ax4.set_ylabel('Mean Absolute Error')
ax4.set_title('Error by CGM Range')

plt.tight_layout()
plt.show()

## 6. Sample Deep Dive

In [ ]:
# Columns to display
display_cols = ['CGM', 'carbs', 'cgm_delta_15min', 'bolus', 'pred_bolus', 'error']
available_display = [c for c in display_cols if c in df.columns]

# Largest Over-predictions (Safety Risk)
print("🔴 TOP 10 OVER-PREDICTIONS (Safety Risk)")
print("=" * 70)
over_preds = df.sort('error', descending=True).head(10)
print(over_preds.select(available_display))

print("\n⚠️ These are cases where model predicted HIGH but true was LOW")

In [ ]:
# Largest Under-predictions (Efficacy Risk)
print("\n🔵 TOP 10 UNDER-PREDICTIONS (Efficacy Risk)")
print("=" * 70)
under_preds = df.sort('error').head(10)
print(under_preds.select(available_display))

print("\n⚠️ These are cases where model predicted LOW but true was HIGH")

## 7. Final Safety Recommendations

In [ ]:
print("\n" + "=" * 70)
print("📋 V6 MODEL FORENSIC REPORT")
print("=" * 70)

print(f"""
GLOBAL METRICS:
   MAE: {mae:.4f}
   RMSE: {rmse:.4f}
   Positive-Only MAE: {pos_mae:.4f}

CLINICAL SAFETY:
   Hypo Zone Mean Pred: {hypo_mean:.4f} U {'✅' if hypo_mean < 0.2 else '⚠️'}
   Safety Zone Safe %: {safety_safe_pct:.1f}% {'✅' if safety_safe_pct > 90 else '⚠️'}
   High Zone Response: {high_mean:.4f} U (Corr: {high_corr:.3f}) {'✅' if high_corr > 0.3 else '⚠️'}

HARD RULES NEEDED IN API:
""")

if hypo_mean >= 0.2:
    print("   ⚠️ RULE 1: IF CGM < 70 → Force bolus = 0")
if safety_safe_pct < 90:
    print("   ⚠️ RULE 2: IF CGM < 100 AND carbs = 0 → Force bolus = 0")
if high_corr < 0.3:
    print("   ⚠️ RULE 3: IF CGM > 250 AND pred < 1.0 → Apply correction floor")

print("\n" + "=" * 70)
print("🏆 FORENSIC ANALYSIS COMPLETE")